# Milestone 1 on Kaggle: scratch CNN and ResNet-50 (probe, fine-tune), 100 % of the data, 3 seeds

Settings: **Accelerator = GPU T4 x2** (only the first GPU is used), **Internet = on**. Run with *Save Version > Save & Run All*.

Every run uses a T4 so that training times are comparable across milestones (recent PyTorch builds may also drop P100 support). The device name is stored in each result file, and the aggregation refuses to mix devices within a group.

1. clones the repository and checks the frozen split with its tests;
2. downloads EuroSAT RGB from the authors' Zenodo record (MIT license) and checks its MD5;
3. runs the three configs, one JSON per run in `/kaggle/working/results`;
4. builds the summary table and zips the results (download `results.zip` from the Output tab).

Runs that already have a result file are skipped, so a session that stops can be rerun.

In [ ]:
import os, shutil, subprocess, torch, timm
print('torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the GPU accelerator')
print('CPU cores:', os.cpu_count(), '| free disk in /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9))

def run(cmd):
    """Run a shell command, stream its output, raise if it fails."""
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
run('rm -rf /tmp/repo && git clone --depth 1 https://github.com/amine-belkacem/eurosat-onboard-transfer.git /tmp/repo')
os.chdir('/tmp/repo')
os.environ['PYTHONPATH'] = 'src'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
run('git log -1 --format="commit %h %s"')
run('python -m pytest -q tests/test_splits.py')

In [ ]:
run('python scripts/download_eurosat.py --data-dir /tmp/data')

In [ ]:
OUT = '/kaggle/working/results'
for cfg in ['cnn_scratch', 'resnet50_probe', 'resnet50_ft']:
    run(f'python scripts/run_experiment.py --config configs/{cfg}.yaml --pcts 100 --seeds 0 1 2 '
        f'--data-root /tmp/data/EuroSAT_RGB/EuroSAT_RGB --out-dir {OUT}')

In [ ]:
run(f'python scripts/summarize.py --results-dir {OUT}')
run('cd /kaggle/working && zip -qr results.zip results && ls -lh results.zip')